In [ ]:
!pip install plotly

In [19]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv(r"C:\Users\Diar\Documents\jupyter\data\Internal_migration.csv")
df.columns = df.columns.str.strip()

name_map = {
    "Almaty_city":   "Almaty city",
    "Astana_city":   "Astana",
    "Shymkent_city": "Shymkent",
    "North_Kaz":     "North Kazakhstan",
    "East_Kaz":      "East Kazakhstan",
    "West_Kaz":      "West Kazakhstan",
}
df["origin"]      = df["region_emmigr"].replace(name_map)
df["destination"] = df["region_immigr"].replace(name_map)
df["year"]        = df["Year"].astype(int)
df["flow"]        = df["N"].astype(int)

all_years = sorted(df["year"].unique())
print(f"Loaded {len(df):,} records · Years: {all_years}")
df.head()

Loaded 3,040 records · Years: [np.int64(2010), np.int64(2015), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,Year,region_emmigr,region_immigr,N,origin,destination,year,flow
0,2025,Almaty,Almaty_city,36729,Almaty,Almaty city,2025,36729
1,2025,Turkestan,Shymkent_city,35436,Turkestan,Shymkent,2025,35436
2,2025,Almaty_city,Almaty,34136,Almaty city,Almaty,2025,34136
3,2025,Shymkent_city,Turkestan,21040,Shymkent,Turkestan,2025,21040
4,2025,Akmola,Astana_city,17326,Akmola,Astana,2025,17326


In [20]:
def get_net_migration(year=None):
    d = df if year is None else df[df["year"] == year]
    arrivals   = d.groupby("destination")["flow"].sum().rename("arrivals")
    departures = d.groupby("origin")["flow"].sum().rename("departures")
    net = pd.DataFrame({"arrivals": arrivals, "departures": departures}).fillna(0)
    net["net"] = net["arrivals"] - net["departures"]
    return net.sort_values("net")

get_net_migration(2023)

,arrivals,departures,net
Turkestan,39486,62597,-23111
Zhambyl,14380,28061,-13681
Zhetysu,10256,19437,-9181
Kyzylorda,7964,15118,-7154
Abay,8598,15346,-6748
Karaganda,12892,17338,-4446
Kostanay,6683,10344,-3661
Pavlodar,7654,11191,-3537
North Kazakhstan,5903,9376,-3473
Akmola,22787,26113,-3326


In [26]:
net23 = get_net_migration(2023)
clrs  = ["#2563EB" if v >= 0 else "#DC2626" for v in net23["net"]]

fig = go.Figure(go.Bar(
    x=net23["net"], y=net23.index,
    orientation="h",
    marker_color=clrs,
    text=[f"{v:+,.0f}" for v in net23["net"]],
    textposition="outside",
    hovertemplate="%{y}: %{x:+,} people<extra></extra>",
))
fig.update_layout(
    title="Net internal migration by region — 2023",
    height=500,
    paper_bgcolor="#F7F8FA",
    plot_bgcolor="#FFFFFF",
    xaxis=dict(showgrid=True, gridcolor="#F3F4F6", zeroline=True),
    margin=dict(l=150, r=80, t=50, b=40),
)
fig.show()

In [27]:
top10 = (
    df[df["year"] == 2023]
    .sort_values("flow", ascending=True)
    .tail(10)
)
top10["label"] = top10["origin"] + " → " + top10["destination"]

fig2 = go.Figure(go.Bar(
    x=top10["flow"], y=top10["label"],
    orientation="h",
    marker_color="#1A1A2E",
    text=[f"{v:,}" for v in top10["flow"]],
    textposition="outside",
))
fig2.update_layout(
    title="Top 10 migration corridors — 2023",
    height=400,
    paper_bgcolor="#F7F8FA",
    plot_bgcolor="#FFFFFF",
    margin=dict(l=200, r=80, t=50, b=40),
)
fig2.show()

In [23]:
cities = {"Astana": "#1A1A2E", "Almaty city": "#2563EB", "Shymkent": "#D97706"}
fig3 = go.Figure()
for city, color in cities.items():
    trend = [get_net_migration(yr).loc[city, "net"] if city in get_net_migration(yr).index else 0 for yr in all_years]
    fig3.add_trace(go.Scatter(
        x=all_years, y=trend,
        mode="lines+markers",
        name=city,
        line=dict(color=color, width=2),
        marker=dict(size=6),
    ))
fig3.update_layout(
    title="Net migration trend — major cities (2010–2025)",
    height=400,
    paper_bgcolor="#F7F8FA",
    plot_bgcolor="#FFFFFF",
    xaxis=dict(tickvals=all_years),
    yaxis=dict(tickformat=","),
)
fig3.show()